# V) Topology optimization


## 1) Geometry

First, we define the geometry, material distribution, and generate the mesh.

In [1]:
pole_pairs = 6
bundles_per_half_slot = 5

# Generate the corresponding mesh
from utils.geometry import machine_mesh
mesh = machine_mesh(p=pole_pairs, 
                    bundles_per_half_slot=bundles_per_half_slot, 
                    hBundle=0.25e-3
                    )

print(f"Generated mesh with {mesh.nv} nodes, {mesh.ne} elements")

Generated mesh with 15477 nodes, 30870 elements


________
## 2) Setup of the physical problem

See notebook [I_Reference_simulation](I_Reference_simulation.ipynb).

### 2.a) Material properties

In [2]:
mur_iron = 1000           # Relative magnetic permeability of iron core
sigma_copper = 5.8e7      # Copper conductivity (S/m)
Br = 1                    # Remanent flux density of magnets (T)


# Define space-varying material coefficients
from ngsolve import pi
mu0 = 4e-7 * pi
nu = mesh.MaterialCF({"core_stator" : 1 / (mu0 * mur_iron)}, default = 1/mu0)     # reluctivity
sigma = mesh.MaterialCF({"slot(.*)_bundle.*" : sigma_copper})                     # conductivity
from utils.physics import magnetization_halbach
Mcplx = mesh.MaterialCF({"rotor" : magnetization_halbach(br = Br, p=pole_pairs)}) # magnetization

### 2.b) Current supply

In [3]:
freq = 1000 # Electrical frequency (Hz)
Jrms = 10e6 # Current density (A/m²)
phi = 150   # load angle (°), chosen to maximize average torque
winding_type = "distributed" # "distributed" or "concentrated"

from ngsolve import Integrate
from utils.supply import phase_current, winding_arrangement, bundle_arrangement

S_bundle = Integrate(1, mesh.Materials("slot11_bundle0"))
Irms = Jrms * S_bundle
phase = phase_current(I_rms=Irms,  load_angle=phi*pi/180)
winding = winding_arrangement(phase, type = winding_type)
bundles = bundle_arrangement(winding, bundles_per_half_slot=bundles_per_half_slot)

### 2.c) Finite element space

In [4]:
curve_order = 2
fem_order = curve_order

from ngsolve import H1, Periodic
fes = Periodic( H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   dirichlet =  "shaft|out", 
                   complex = True),  [-1]*7 )
print(f"Number of degrees of freedom of the FE space: {fes.ndof}")

Number of degrees of freedom of the FE space: 61823


_________
## 3) Point-wise optimization of the conductivity

Too high conductivity leads to high AC losses, while too low conductivity leads to high DC losses. There is an optimum point that can be found. An optimization of the whole conductivity of a single material coil was conducted in [3_single_material_optimization](3_single_material_optimization.ipynb), and the optimization of the conductivity of each bundle independantly was performed in [4_multi_material_optimization](4_multi_material_optimization.ipynb).

However, additive manufacturing may enable even more freedom, so that the conductivity can take any value in the whole cross section.

### 3.a) Setup of the optimization problem

In [5]:
# Problem parameters
sigma0 = sigma_copper         # initial conductivity
sigma_min = sigma_copper/10   # minimum admissible conductivity
sigma_max = sigma_copper      # maximum admissible conductivity

# Conductivity space
from ngsolve import L2, GridFunction
fes_sigma = fes_sigma = H1(mesh, order = 1, definedon = "slot(.*)_bundle.*")
x0 = GridFunction(fes_sigma)
x0.Set(sigma0)

In [6]:
# Definition of the functions of interest

from utils.physics import solve_magnetoharmonic, joule_losses
def state_function(conductivity):
    """ Returns the solution of the magnetoharmonic problem for a given conductivity value """
    result = solve_magnetoharmonic(fes=fes,
                                   reluctivity=nu,
                                   conductivity=conductivity,
                                   magnetization=Mcplx,
                                   frequency=freq,
                                   supply=bundles
                                   )
    return result

def objective_function(result):
    """ Total Joule losses to minimize """
    return joule_losses(result)

In [7]:
# Algorithm parameters

from copy import copy
from numpy import sign

def descent(grad):
    """ Extract descent direction from the gradient """
    descent = copy(grad)
    descent.vec.data = - 1e7 * sign(grad.vec)
    return descent

# Derivative
from utils.optimization import d_joule_losses
from ngsolve import InnerProduct, CoefficientFunction as CF

def grad_joule_losses_local(result):
    """ Compute the gradient of the objective function w.r.t the conductivity value at each point """
    fes_sigma = result["info"]["conductivity"].space
    dx_global = GridFunction(fes_sigma)
    dx_global.Set(1)
    df = d_joule_losses(result, dx_global)
    grad_f = GridFunction(fes_sigma)
    grad_f.vec.data = df.vec
    return grad_f

# Derivative
def grad_joule_losses_bundle(result):
    """ Compute the gradient of the objective function w.r.t the conductivity value of each bundle """
    fes_sigma = result["info"]["conductivity"].space
    dx_global = GridFunction(fes_sigma)
    dx_global.Set(1)
    df = d_joule_losses(result, dx_global)
    # we want now to provide a conductivity field representing the steepest admissible ascent direction (we call this the "gradient")
    dfdx = CF(0)
    for bundle in result["info"]["supply"].keys():
        dx = GridFunction(fes_sigma)
        dx.Set(mesh.MaterialCF({bundle : 1})) # dx is a unit perturbation of sigma in a single bundle
        dfdx += InnerProduct(df.vec, dx.vec) * copy(dx)  
    grad_f = GridFunction(fes_sigma)
    grad_f.Set(dfdx.Compile())
    return grad_f

### 3.b) Optimization loop

In [ ]:
from utils.optimization import gradient_descent

results_topopt_sigma = gradient_descent(state=state_function,
                                 objective = objective_function,
                                 d_objective = grad_joule_losses_local,
                                 x0 = x0,
                                 x_min = sigma_min,
                                 x_max = sigma_max,
                                 descent = descent)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

it 1 ✅| obj = 3.383e+04 | step = 1.00e+00 | crit = 1.00e+00
it 2 ✅| obj = 3.074e+04 | step = 1.20e+00 | crit = 1.06e+00
it 3 ✅| obj = 2.579e+04 | step = 1.44e+00 | crit = 9.98e-01
it 4 ✅| obj = 1.797e+04 | step = 1.73e+00 | crit = 9.77e-01
it 5 ✅| obj = 7.734e+03 | step = 2.07e+00 | crit = 3.34e-01
it 6 ✅| obj = 7.578e+03 | step = 2.49e+00 | crit = 3.22e-01
it 7 ✅| obj = 7.528e+03 | step = 2.99e+00 | crit = 3.41e-01
it 8 ✅| obj = 7.508e+03 | step = 3.58e+00 | crit = 3.03e-01
it 9 ✅| obj = 7.501e+03 | step = 4.30e+00 | crit = 3.28e-01
it 10 ✅| obj = 7.500e+03 | step = 5.16e+00 | crit = 3.07e-01
it 11 ✅| obj = 7.497e+03 | step = 6.19e+00 | crit = 2.92e-01
it 12 ❌| obj = 7.498e+03 | step = 3.10e+00 | crit = 2.92e-01
it 13 ✅| obj = 7.496e+03 | step = 3.72e+00 | crit = 3.91e-01
it 14 ✅| obj = 7.495e+03 | step = 4.46e+00 | crit = 2.64e-01
it 15 ❌| obj = 7.497e+03 | step = 2.23e+00 | crit = 2.64e-01
it 16 ✅| obj = 7.495e+03 | step = 2.67e+00 | crit = 3.43e-01
it 17 ✅| obj = 7.494e+03 | step =

### 3.c) Results

In [ ]:
# Display the optimal conductivity distribution
from numpy import nan
from ngsolve.webgui import Draw
result_optim_global = results_topopt_sigma["solution"][-1] + mesh.MaterialCF({"slot.*_bundle.*" : 0}, default = nan)
Draw(result_optim_global, results_topopt_sigma["solution"][-1].space.mesh,
     settings = {"Objects" : {"Wireframe" : False}, "Colormap" : {"ncolors" : 32}},
     filename = "scenes/optim/topopt/sigma_topopt.html",
     min = sigma_min, max = sigma_max,
     )

In [ ]:
# Display convergence
import matplotlib.pyplot as plt
sigma_global_opt = results_topopt_sigma["solution"][-1].vec.FV().NumPy()[0]
print(f"Optimal global conductivity = {sigma_global_opt :.2e} S/m")
print(f"=> Joule losses = {results_topopt_sigma['objective'][-1]:.2e} W/m")
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(results_topopt_sigma["objective"], 'b-')
ax2.semilogy(results_topopt_sigma["criterion"], 'r-')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Joule losses (W/m)', color='b')
ax2.set_ylabel('Stop criterion', color='r')
plt.title(f"Convergence, Pj = {results_topopt_sigma['objective'][-1]:.2e} W/m")
plt.savefig("scenes/optim/topopt/convergence_topopt.pdf")
plt.show()

In [ ]:
results_topopt_sigma_DC = solve_magnetoharmonic(fes=fes,
                                reluctivity=nu,
                                conductivity=results_topopt_sigma["solution"][-1],
                                magnetization=Mcplx,
                                frequency=1e-5,
                                supply=bundles,
                                   )


total_losses_optim = results_topopt_sigma["objective"][-1]
DC_losses_optim = joule_losses(results_topopt_sigma_DC)
AC_losses_optim = total_losses_optim - DC_losses_optim

print(f"Optimized total Joule losses: {total_losses_optim:.2e} W/m")
print(f"Optimized DC Joule losses: {DC_losses_optim:.2e} W/m ({DC_losses_optim/total_losses_optim*100:.1f} %)")
print(f"Optimized AC Joule losses: {AC_losses_optim:.2e} W/m ({AC_losses_optim/total_losses_optim*100:.1f} %)")

___
## Summary

We find the following results after a gradient descent at $f = 1\;\text{kHz}$, for a topology optimization (conductivity has a different value at each point):

| Total losses at 1kHz | $P_{topopt} = 7.49\times 10^3 \; \text{W/m}$ | 100% |
|---|---|------|
| DC losses | $P_{DC-topopt} = 2.4\times 10^3  \; \text{W/m}$ | XX %   |
| AC losses | $P_{AC-topopt} = XX\times 10^3  \; \text{W/m}$ | XXX %   |

Again, we don't have a perfect 50/50 split between AC and DC losses, however, the conductivity distribution looks close to optimality from the convergence curves.

For now, that's the best we can do!